<a href="https://colab.research.google.com/github/Kevan123/BB-Traffic-Analysis-Challenge-Zindi-/blob/main/Barbados_Traffic_Analysis_Challenge_Zindi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set Up Environment, Load Variables, Prep Data, Intial Models (XGB)

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

# =========================
# Config
# =========================
DATA_DIR = Path(".")  # adjust if needed

TRAIN_FILE = "/content/Train.csv"
TEST_INPUT_FILE = "/content/TestInputSegments.csv"
SAMPLE_SUB_FILE = "/content/SampleSubmission.csv"
SUBMISSION_OUT = "submission.csv"

HIST_LEN = 15      # length of history window
EMBARGO = 2        # 2-minute embargo before prediction

USE_TEST_INPUT_AS_TRAIN = True  # you can toggle this if you want "pure" Train only



In [4]:
# =========================
# Load data
# =========================

train = pd.read_csv(TRAIN_FILE)
test_in = pd.read_csv(TEST_INPUT_FILE)
sample_sub = pd.read_csv(SAMPLE_SUB_FILE)

# Basic sanity
assert {"view_label", "time_segment_id", "signaling"}.issubset(train.columns)
assert {"view_label", "time_segment_id", "signaling"}.issubset(test_in.columns)

In [5]:
train.head()

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


In [6]:
# =========================
# Label & feature encoders
# =========================

CONG_MAP = {
    "free flowing": 0,
    "light delay": 1,
    "moderate delay": 2,
    "heavy delay": 3,
}
CONG_MAP_INV = {v: k for k, v in CONG_MAP.items()}

SIGNAL_MAP = {
    "none": 0,
    "low": 1,
    "medium": 2,
    "high": 3,
}



In [7]:
# =========================
# Preprocess & combine
# =========================

def prep_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Encode signaling
    out["signal_enc"] = out["signaling"].map(SIGNAL_MAP).fillna(0).astype(int)

    # Encode labels if present
    if "congestion_enter_rating" in out.columns:
        out["enter_label_int"] = out["congestion_enter_rating"].map(CONG_MAP)
    if "congestion_exit_rating" in out.columns:
        out["exit_label_int"] = out["congestion_exit_rating"].map(CONG_MAP)

    # Parse timestamps if present (used for potential time-of-day features)
    if "datetimestamp_start" in out.columns:
        out["datetimestamp_start"] = pd.to_datetime(
            out["datetimestamp_start"], errors="coerce"
        )

    return out


train = prep_df(train)
test_in = prep_df(test_in)

# Tag phases
train["phase"] = train.get("cycle_phase", "train")
test_in["phase"] = test_in.get("cycle_phase", "test_input_15")

# In practice, all_df is the historical universe available at inference time.
all_df = pd.concat([train, test_in], ignore_index=True)

# Sort in true temporal order: by camera then by segment id
all_df = all_df.sort_values(["view_label", "time_segment_id"]).reset_index(drop=True)

In [8]:
# =========================
# Feature construction
# =========================

FEATURE_COLS = None  # will populate after building training matrix


def make_feature_vector(
    hist_df: pd.DataFrame,
    hist_len: int = HIST_LEN,
) -> dict:
    """
    Build features from a history window (already filtered to one view_label
    and sorted by time_segment_id). Real-time safe: uses ONLY:
      - signaling history
      - simple aggregates
      - (optionally) time-of-day of last observed segment
    """
    assert len(hist_df) > 0

    # Ensure sorted
    hist_df = hist_df.sort_values("time_segment_id")

    sig = hist_df["signal_enc"].values
    # If shorter than hist_len, left-pad with earliest value (real-time safe)
    if len(sig) < hist_len:
        pad_val = sig[0]
        sig = np.concatenate([np.full(hist_len - len(sig), pad_val), sig])
    else:
        sig = sig[-hist_len:]  # take most recent hist_len

    feats = {}

    # Raw lags: lag_1 = most recent pre-embargo, lag_hist_len = oldest in window
    # (this ordering is arbitrary but consistent)
    for i in range(hist_len):
        feats[f"sig_lag_{i+1}"] = int(sig[-(i+1)])

    # Aggregates
    feats["sig_mean_15"] = float(sig.mean())
    feats["sig_std_15"] = float(sig.std())
    feats["sig_min_15"] = float(sig.min())
    feats["sig_max_15"] = float(sig.max())

    # Time-of-day features from last history segment (if available)
    if "datetimestamp_start" in hist_df.columns and hist_df["datetimestamp_start"].notna().any():
        last_ts = hist_df["datetimestamp_start"].iloc[-1]
        if pd.notna(last_ts):
            hour = last_ts.hour + last_ts.minute / 60.0
            feats["tod_sin"] = np.sin(2 * np.pi * hour / 24.0)
            feats["tod_cos"] = np.cos(2 * np.pi * hour / 24.0)
        else:
            feats["tod_sin"] = 0.0
            feats["tod_cos"] = 0.0
    else:
        feats["tod_sin"] = 0.0
        feats["tod_cos"] = 0.0

    return feats


def build_training_matrix(
    all_df: pd.DataFrame,
    label_col: str,
    phases_for_training=("train", "test_input_15"),
    hist_len: int = HIST_LEN,
    embargo: int = EMBARGO,
):
    """
    Build (X, y) for a given label (enter or exit) using a rolling
    15-min history + 2-min embargo.

    For each target row t:
      - Identify history window: [t_id - embargo - hist_len, ..., t_id - embargo - 1]
      - Use ONLY rows from all_df with:
          same view_label
          time_segment_id in that range
      - Use target only if its phase is in phases_for_training.
    """
    rows = []
    ys = []

    # Pre-index for speed
    # group by camera
    grouped = all_df.groupby("view_label", group_keys=False)

    for view_label, g in grouped:
        g = g.sort_values("time_segment_id")

        # Build lookup by time_segment_id for that view
        # (assumes unique per view, which holds in this dataset)
        idx_by_ts = {ts: idx for idx, ts in zip(g.index, g["time_segment_id"])}

        for idx, row in g.iterrows():
            phase = row["phase"]
            if phase not in phases_for_training:
                continue

            target_ts = int(row["time_segment_id"])

            # Define history range
            start_ts = target_ts - embargo - hist_len
            end_ts = target_ts - embargo - 1

            if start_ts < 0:
                # Not enough history at the very beginning; skip
                continue

            # Collect history rows (within same camera + time range)
            hist_mask = (g["time_segment_id"] >= start_ts) & (g["time_segment_id"] <= end_ts)
            hist_df = g.loc[hist_mask]

            if len(hist_df) == 0:
                continue  # no usable history

            feats = make_feature_vector(hist_df, hist_len=hist_len)

            # Keep camera as categorical if you want; here we'll one-hot-ish
            feats[f"cam_{view_label}"] = 1

            label_val = row[label_col]
            # skip if label missing
            if np.isnan(label_val):
                continue

            rows.append(feats)
            ys.append(int(label_val))

    if not rows:
        raise RuntimeError("No training rows constructed. Check ranges/config.")

    X = pd.DataFrame(rows).fillna(0)

    # Ensure camera columns exist for all views (for consistency with test)
    for v in all_df["view_label"].unique():
        col = f"cam_{v}"
        if col not in X.columns:
            X[col] = 0

    # Sort columns for consistency
    X = X.reindex(sorted(X.columns), axis=1)

    y = np.array(ys, dtype=int)

    return X, y


# Map textual labels to ints in all_df
all_df["enter_label_int"] = all_df["congestion_enter_rating"].map(CONG_MAP)
all_df["exit_label_int"] = all_df["congestion_exit_rating"].map(CONG_MAP)

In [9]:
# =========================
# Build training sets
# =========================

phases = ["train"] + (["test_input_15"] if USE_TEST_INPUT_AS_TRAIN else [])

X_enter, y_enter = build_training_matrix(
    all_df,
    label_col="enter_label_int",
    phases_for_training=phases,
    hist_len=HIST_LEN,
    embargo=EMBARGO,
)

X_exit, y_exit = build_training_matrix(
    all_df,
    label_col="exit_label_int",
    phases_for_training=phases,
    hist_len=HIST_LEN,
    embargo=EMBARGO,
)

# Use same feature space for both
FEATURE_COLS = sorted(set(X_enter.columns) | set(X_exit.columns))
X_enter = X_enter.reindex(columns=FEATURE_COLS, fill_value=0)
X_exit = X_exit.reindex(columns=FEATURE_COLS, fill_value=0)


In [10]:
# =========================
# Train models (XGBoost → fallback to RF)
# =========================

try:
    from xgboost import XGBClassifier

    enter_model = XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        max_depth=4,
        n_estimators=400,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
    )
    exit_model = XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        max_depth=4,
        n_estimators=400,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
    )

except ImportError:
    # Simple fallback if xgboost is not installed
    from sklearn.ensemble import RandomForestClassifier

    enter_model = RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )
    exit_model = RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )

# Fit
enter_model.fit(X_enter, y_enter)
exit_model.fit(X_exit, y_exit)

RandomForestClassifier(max_depth=8, n_estimators=400, n_jobs=-1,
                       random_state=42)

In [11]:
# =========================
# Build test features for SampleSubmission IDs 1
# =========================

# Parse IDs: time_segment_<id>_<view_label>_congestion_(enter|exit)_rating
parsed = sample_sub["ID"].str.extract(
    r"time_segment_(\d+)_([A-Za-z0-9 #]+)_congestion_(enter|exit)_rating"
)
parsed.columns = ["time_segment_id", "view_label", "target_kind"]

sample_sub = sample_sub.copy()
sample_sub["time_segment_id"] = parsed["time_segment_id"].astype(int)
sample_sub["view_label"] = parsed["view_label"]
sample_sub["target_kind"] = parsed["target_kind"]

# Pre-group all_df by camera for efficient lookup
all_grouped = {
    v: g.sort_values("time_segment_id").reset_index(drop=True)
    for v, g in all_df.groupby("view_label")
}


def build_features_for_submission(
    sample_df: pd.DataFrame,
    all_grouped: dict,
    hist_len: int = HIST_LEN,
    embargo: int = EMBARGO,
) -> pd.DataFrame:
    """
    For each row in sample_df, build a feature vector consistent with training:
      - for target at time_segment_id = T, camera = C:
        use history [T - embargo - hist_len, ..., T - embargo - 1] from same camera.
      - if not enough history, left-pad safely.
    """
    feat_rows = []

    for _, row in sample_df.iterrows():
        ts = int(row["time_segment_id"])
        view = row["view_label"]

        if view not in all_grouped:
            # unseen camera: create dummy features
            hist_df = pd.DataFrame({"signal_enc": [0]})
        else:
            g = all_grouped[view]
            start_ts = ts - embargo - hist_len
            end_ts = ts - embargo - 1

            hist_mask = (g["time_segment_id"] >= start_ts) & (g["time_segment_id"] <= end_ts)
            hist_df = g.loc[hist_mask]

            if len(hist_df) == 0:
                # if no history at all (edge case), create a fake neutral history
                hist_df = pd.DataFrame({"signal_enc": [0]})

        feats = make_feature_vector(hist_df, hist_len=hist_len)

        # add camera one-hot (same as training)
        # we add only one, the rest will be created/zeroed later
        feats[f"cam_{view}"] = 1

        feat_rows.append(feats)

    X_test = pd.DataFrame(feat_rows).fillna(0)

    # Ensure all train feature columns exist
    for col in FEATURE_COLS:
        if col not in X_test.columns:
            X_test[col] = 0

    # Extra cols in X_test that weren't in training: drop them
    X_test = X_test[FEATURE_COLS]

    return X_test


X_sub = build_features_for_submission(sample_sub, all_grouped, HIST_LEN, EMBARGO)


# =========================
# Predict & build submission
# =========================

# Predict class indices and probabilities for ALL rows in X_sub
enter_pred_int = enter_model.predict(X_sub)
exit_pred_int = exit_model.predict(X_sub)

# predict_proba gives [n_rows, n_classes] with probs in same class order as training labels (0,1,2,3)
enter_proba = enter_model.predict_proba(X_sub)
exit_proba = exit_model.predict_proba(X_sub)

# Map to label strings
enter_labels = pd.Series(enter_pred_int).map(CONG_MAP_INV)
exit_labels = pd.Series(exit_pred_int).map(CONG_MAP_INV)

# Fill Targets based on whether ID is "enter" or "exit"
targets = []
target_acc = []
for i, row in sample_sub.iterrows():
    if row["target_kind"] == "enter":
        # Get the predicted class label string
        cls_label = enter_labels.iloc[i]
    else:
        # Get the predicted class label string
        cls_label = exit_labels.iloc[i]

    targets.append(cls_label)
    target_acc.append(cls_label) # Set Target_Accuracy to the predicted class label

submission = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets,
    "Target_Accuracy": target_acc,
})

submission.to_csv(SUBMISSION_OUT, index=False)

print(f"Submission file written to: {SUBMISSION_OUT}")

Submission file written to: submission.csv


In [12]:
from sklearn.metrics import f1_score, accuracy_score

# Evaluate 'enter' model
y_enter_pred = enter_model.predict(X_enter)
f1_enter = f1_score(y_enter, y_enter_pred, average='weighted')
accuracy_enter = accuracy_score(y_enter, y_enter_pred)

print(f"Enter Model - F1 Score: {f1_enter:.4f}")
print(f"Enter Model - Accuracy: {accuracy_enter:.4f}")

# Evaluate 'exit' model
y_exit_pred = exit_model.predict(X_exit)
f1_exit = f1_score(y_exit, y_exit_pred, average='weighted')
accuracy_exit = accuracy_score(y_exit, y_exit_pred)

print(f"\nExit Model - F1 Score: {f1_exit:.4f}")
print(f"Exit Model - Accuracy: {accuracy_exit:.4f}")

Enter Model - F1 Score: 0.6002
Enter Model - Accuracy: 0.6845

Exit Model - F1 Score: 0.9345
Exit Model - Accuracy: 0.9560


## Additional Model Builds and Comparisons
The following section several models will be built and compared.


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Train Logistic Regression models
lr_enter = LogisticRegression(max_iter=1000, random_state=42)
lr_enter.fit(X_enter, y_enter)

lr_exit = LogisticRegression(max_iter=1000, random_state=42)
lr_exit.fit(X_exit, y_exit)

# Train SVC models
svc_enter = SVC(probability=True, random_state=42)
svc_enter.fit(X_enter, y_enter)

svc_exit = SVC(probability=True, random_state=42)
svc_exit.fit(X_exit, y_exit)

# Train RandomForestClassifier models
rf_enter = RandomForestClassifier(random_state=42)
rf_enter.fit(X_enter, y_enter)

rf_exit = RandomForestClassifier(random_state=42)
rf_exit.fit(X_exit, y_exit)

RandomForestClassifier(random_state=42)

In [14]:
# Predict and generate submission for Logistic Regression
lr_enter_pred = lr_enter.predict(X_sub)
lr_exit_pred = lr_exit.predict(X_sub)
lr_enter_proba = lr_enter.predict_proba(X_sub)
lr_exit_proba = lr_exit.predict_proba(X_sub)

targets_lr = []
target_acc_lr = []

for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        cls_int = int(lr_enter_pred[i])
        cls_label = CONG_MAP_INV[cls_int]
        # cls_conf = float(lr_enter_proba[i, cls_int]) # Use probability
    else:
        cls_int = int(lr_exit_pred[i])
        cls_label = CONG_MAP_INV[cls_int]
        # cls_conf = float(lr_exit_proba[i, cls_int]) # Use probability

    targets_lr.append(cls_label)
    target_acc_lr.append(cls_label) # Set Target_Accuracy to the predicted class label

submission_lr = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_lr,
    "Target_Accuracy": target_acc_lr,
})
submission_lr.to_csv("submission_lr.csv", index=False)
print("Submission file written to: submission_lr.csv")

# Predict and generate submission for SVC
svc_enter_pred = svc_enter.predict(X_sub)
svc_exit_pred = svc_exit.predict(X_sub)
svc_enter_proba = svc_enter.predict_proba(X_sub)
svc_exit_proba = svc_exit.predict_proba(X_sub)

targets_svc = []
target_acc_svc = []

for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        cls_int = int(svc_enter_pred[i])
        cls_label = CONG_MAP_INV[cls_int]
        # cls_conf = float(svc_enter_proba[i, cls_int]) # Use probability
    else:
        cls_int = int(svc_exit_pred[i])
        cls_label = CONG_MAP_INV[cls_int]
        # cls_conf = float(svc_exit_proba[i, cls_int]) # Use probability

    targets_svc.append(cls_label)
    target_acc_svc.append(cls_label) # Set Target_Accuracy to the predicted class label

submission_svc = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_svc,
    "Target_Accuracy": target_acc_svc,
})
submission_svc.to_csv("submission_svc.csv", index=False)
print("Submission file written to: submission_svc.csv")

# Predict and generate submission for RandomForestClassifier
rf_enter_pred = rf_enter.predict(X_sub)
rf_exit_pred = rf_exit.predict(X_sub)
rf_enter_proba = rf_enter.predict_proba(X_sub)
rf_exit_proba = rf_exit.predict_proba(X_sub)

targets_rf = []
target_acc_rf = []

for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        cls_int = int(rf_enter_pred[i])
        cls_label = CONG_MAP_INV[cls_int]
        # cls_conf = float(rf_enter_proba[i, cls_int]) # Use probability
    else:
        cls_int = int(rf_exit_pred[i])
        cls_label = CONG_MAP_INV[cls_int]
        # cls_conf = float(rf_exit_proba[i, cls_int]) # Use probability

    targets_rf.append(cls_label)
    target_acc_rf.append(cls_label) # Set Target_Accuracy to the predicted class label

submission_rf = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_rf,
    "Target_Accuracy": target_acc_rf,
})
submission_rf.to_csv("submission_rf.csv", index=False)
print("Submission file written to: submission_rf.csv")

Submission file written to: submission_lr.csv
Submission file written to: submission_svc.csv
Submission file written to: submission_rf.csv


In [15]:
from sklearn.metrics import f1_score, accuracy_score

# Evaluate Logistic Regression models
lr_enter_pred = lr_enter.predict(X_enter)
f1_lr_enter = f1_score(y_enter, lr_enter_pred, average='weighted')
accuracy_lr_enter = accuracy_score(y_enter, lr_enter_pred)

lr_exit_pred = lr_exit.predict(X_exit)
f1_lr_exit = f1_score(y_exit, lr_exit_pred, average='weighted')
accuracy_lr_exit = accuracy_score(y_exit, lr_exit_pred)

print(f"Logistic Regression Enter Model - F1 Score: {f1_lr_enter:.4f}")
print(f"Logistic Regression Enter Model - Accuracy: {accuracy_lr_enter:.4f}")
print(f"Logistic Regression Exit Model - F1 Score: {f1_lr_exit:.4f}")
print(f"Logistic Regression Exit Model - Accuracy: {accuracy_lr_exit:.4f}")

# Evaluate SVC models
svc_enter_pred = svc_enter.predict(X_enter)
f1_svc_enter = f1_score(y_enter, svc_enter_pred, average='weighted')
accuracy_svc_enter = accuracy_score(y_enter, svc_enter_pred)

svc_exit_pred = svc_exit.predict(X_exit)
f1_svc_exit = f1_score(y_exit, svc_exit_pred, average='weighted')
accuracy_svc_exit = accuracy_score(y_exit, svc_exit_pred)

print(f"\nSVC Enter Model - F1 Score: {f1_svc_enter:.4f}")
print(f"SVC Enter Model - Accuracy: {accuracy_svc_enter:.4f}")
print(f"SVC Exit Model - F1 Score: {f1_svc_exit:.4f}")
print(f"SVC Exit Model - Accuracy: {accuracy_svc_exit:.4f}")

# Evaluate RandomForestClassifier models
rf_enter_pred = rf_enter.predict(X_enter)
f1_rf_enter = f1_score(y_enter, rf_enter_pred, average='weighted')
accuracy_rf_enter = accuracy_score(y_enter, rf_enter_pred)

rf_exit_pred = rf_exit.predict(X_exit)
f1_rf_exit = f1_score(y_exit, rf_exit_pred, average='weighted')
accuracy_rf_exit = accuracy_score(y_exit, rf_exit_pred)

print(f"\nRandomForestClassifier Enter Model - F1 Score: {f1_rf_enter:.4f}")
print(f"RandomForestClassifier Enter Model - Accuracy: {accuracy_rf_enter:.4f}")
print(f"RandomForestClassifier Exit Model - F1 Score: {f1_rf_exit:.4f}")
print(f"RandomForestClassifier Exit Model - Accuracy: {accuracy_rf_exit:.4f}")

Logistic Regression Enter Model - F1 Score: 0.5835
Logistic Regression Enter Model - Accuracy: 0.6463
Logistic Regression Exit Model - F1 Score: 0.9345
Logistic Regression Exit Model - Accuracy: 0.9560

SVC Enter Model - F1 Score: 0.6447
SVC Enter Model - Accuracy: 0.7055
SVC Exit Model - F1 Score: 0.9345
SVC Exit Model - Accuracy: 0.9560

RandomForestClassifier Enter Model - F1 Score: 0.9977
RandomForestClassifier Enter Model - Accuracy: 0.9977
RandomForestClassifier Exit Model - F1 Score: 0.9998
RandomForestClassifier Exit Model - Accuracy: 0.9998


In [16]:
evaluation_results = pd.DataFrame({
    'Model': [
        'XGBoost', 'XGBoost',
        'Logistic Regression', 'Logistic Regression',
        'SVC', 'SVC',
        'RandomForestClassifier', 'RandomForestClassifier'
    ],
    'Target': [
        'Enter', 'Exit',
        'Enter', 'Exit',
        'Enter', 'Exit',
        'Enter', 'Exit'
    ],
    'F1 Score': [
        f1_enter, f1_exit,
        f1_lr_enter, f1_lr_exit,
        f1_svc_enter, f1_svc_exit,
        f1_rf_enter, f1_rf_exit
    ],
    'Accuracy': [
        accuracy_enter, accuracy_exit,
        accuracy_lr_enter, accuracy_lr_exit,
        accuracy_svc_enter, accuracy_svc_exit,
        accuracy_rf_enter, accuracy_rf_exit
    ]
})

display(evaluation_results)

,Model,Target,F1 Score,Accuracy
0,XGBoost,Enter,0.600159,0.684524
1,XGBoost,Exit,0.934535,0.956027
2,Logistic Regression,Enter,0.583533,0.646343
3,Logistic Regression,Exit,0.934535,0.956027
4,SVC,Enter,0.644747,0.705491
5,SVC,Exit,0.934535,0.956027
6,RandomForestClassifier,Enter,0.997746,0.997748
7,RandomForestClassifier,Exit,0.999839,0.999839


### Define hyperparameter search space for XGB

Range of values for the hyperparameters to be tuned for the XGBoost models.


In [17]:
param_grid_xgb = {
    'max_depth': [3, 4, 5],
    'n_estimators': [300, 400, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
}

### Perform hyperparameter tuning for XGB
Use the technique GridSearchCV to find the best hyperparameters for both the 'enter' and 'exit' XGBoost models using cross-validation.


In [18]:
!pip install xgboost

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier # Explicitly import XGBClassifier

# Re-instantiate the base XGBoost models for GridSearchCV
# Using the parameters from the initial XGBoost model definition (cell Af64mXqYndvK)
enter_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    max_depth=4,
    n_estimators=400,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)
exit_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    max_depth=4,
    n_estimators=400,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

# Instantiate GridSearchCV for the 'enter' XGBoost model
grid_search_enter = GridSearchCV(estimator=enter_model, param_grid=param_grid_xgb, scoring='f1_weighted', cv=3, n_jobs=-1)

# Fit GridSearchCV to the 'enter' training data
grid_search_enter.fit(X_enter, y_enter)

# Instantiate GridSearchCV for the 'exit' XGBoost model
grid_search_exit = GridSearchCV(estimator=exit_model, param_grid=param_grid_xgb, scoring='f1_weighted', cv=3, n_jobs=-1)

# Fit GridSearchCV to the 'exit' training data
grid_search_exit.fit(X_exit, y_exit)

# Store the best hyperparameters
best_params_enter = grid_search_enter.best_params_
best_params_exit = grid_search_exit.best_params_

print("Best hyperparameters for Enter model:")
print(best_params_enter)

print("\nBest hyperparameters for Exit model:")
print(best_params_exit)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 1.3 MB/s eta 0:00:00
Best hyperparameters for Enter model:
{'colsample_bytree': 0.9, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.8}

Best hyperparameters for Exit model:
{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}


### Retrain optimized xgboost models

Train the final XGBoost models using the entire training data with the best hyperparameters found during tuning.


In [19]:
from xgboost import XGBClassifier

# Instantiate and train the final 'enter' XGBoost model with best hyperparameters
final_enter_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    **best_params_enter
)
final_enter_model.fit(X_enter, y_enter)

# Instantiate and train the final 'exit' XGBoost model with best hyperparameters
final_exit_model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    **best_params_exit
)
final_exit_model.fit(X_exit, y_exit)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1, num_class=4, ...)

### Reevaluate optimized xgboost models

Evaluate the performance of the optimized XGBoost models on the training data using F1 score and accuracy.


In [20]:
# Evaluate optimized 'enter' model
y_enter_pred_optimized = final_enter_model.predict(X_enter)
f1_enter_optimized = f1_score(y_enter, y_enter_pred_optimized, average='weighted')
accuracy_enter_optimized = accuracy_score(y_enter, y_enter_pred_optimized)

print(f"Optimized Enter Model - F1 Score: {f1_enter_optimized:.4f}")
print(f"Optimized Enter Model - Accuracy: {accuracy_enter_optimized:.4f}")

# Evaluate optimized 'exit' model
y_exit_pred_optimized = final_exit_model.predict(X_exit)
f1_exit_optimized = f1_score(y_exit, y_exit_pred_optimized, average='weighted')
accuracy_exit_optimized = accuracy_score(y_exit, y_exit_pred_optimized)

print(f"\nOptimized Exit Model - F1 Score: {f1_exit_optimized:.4f}")
print(f"Optimized Exit Model - Accuracy: {accuracy_exit_optimized:.4f}")

Optimized Enter Model - F1 Score: 0.8947
Optimized Enter Model - Accuracy: 0.8982

Optimized Exit Model - F1 Score: 0.9345
Optimized Exit Model - Accuracy: 0.9560


### Generate submission file with optimized models

Use the optimized XGBoost models to predict on the test data and generate a new submission file.


In [21]:
# Predict class probabilities for ALL rows in X_sub using optimized models
enter_proba_optimized = final_enter_model.predict_proba(X_sub)
exit_proba_optimized = final_exit_model.predict_proba(X_sub)

targets_optimized = []
target_acc_optimized = []

for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        # use optimized enter model outputs for this row
        # Find the index of the class with the highest probability
        cls_int = int(np.argmax(enter_proba_optimized[i]))
        # Get the corresponding label string
        cls_label = CONG_MAP_INV[cls_int]
        # Get the confidence (probability) of the predicted class
        # cls_conf = float(enter_proba_optimized[i, cls_int])
    else:
        # use optimized exit model outputs for this row
        # Find the index of the class with the highest probability
        cls_int = int(np.argmax(exit_proba_optimized[i]))
        # Get the corresponding label string
        cls_label = CONG_MAP_INV[cls_int]
        # Get the confidence (probability) of the predicted class
        # cls_conf = float(exit_proba_optimized[i, cls_int])

    targets_optimized.append(cls_label)
    target_acc_optimized.append(cls_label) # Set Target_Accuracy to the predicted class label

submission_optimized = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_optimized,
    "Target_Accuracy": target_acc_optimized,
})

submission_optimized.to_csv("submission_optimized.csv", index=False)
print("Submission file written to: submission_optimized.csv")

Submission file written to: submission_optimized.csv


In [22]:
#new_results = pd.DataFrame({
#    'Model': ['Optimized XGBoost', 'Optimized XGBoost'],
#    'Target': ['Enter', 'Exit'],
#    'F1 Score': [f1_enter_optimized, f1_exit_optimized],
#    'Accuracy': [accuracy_enter_optimized, accuracy_exit_optimized]
#})

#evaluation_results = pd.concat([evaluation_results, new_results], ignore_index=True)

#display(evaluation_results)

### Recreation of the submission files
Recreating the `generate_submission_file` function to append `cls_label` to `target_acc` instead of `cls_conf`.

In [23]:
def generate_submission_file(enter_model, exit_model, X_sub, sample_sub, cong_map_inv, filename):
    """
    Generates a submission file with predicted class labels and their probabilities.

    Args:
        enter_model: Trained model for 'enter' congestion prediction.
        exit_model: Trained model for 'exit' congestion prediction.
        X_sub (pd.DataFrame): Test features DataFrame.
        sample_sub (pd.DataFrame): Sample submission DataFrame with parsed IDs.
        cong_map_inv (dict): Inverse mapping of congestion labels to integers.
        filename (str): Name of the output submission file.
    """
    # Predict class probabilities for ALL rows in X_sub
    enter_proba = enter_model.predict_proba(X_sub)
    exit_proba = exit_model.predict_proba(X_sub)

    targets = []
    target_acc = []

    for i, row in sample_sub.iterrows():
        kind = row["target_kind"]

        if kind == "enter":
            # Find the index of the class with the highest probability
            cls_int = int(np.argmax(enter_proba[i]))
            # Get the corresponding label string
            cls_label = cong_map_inv[cls_int]
            # Get the confidence (probability) of the predicted class
            # cls_conf = float(enter_proba[i, cls_int]) # Original line
        else:
            # Find the index of the class with the highest probability
            cls_int = int(np.argmax(exit_proba[i]))
            # Get the corresponding label string
            cls_label = cong_map_inv[cls_int]
            # Get the confidence (probability) of the predicted class
            # cls_conf = float(exit_proba[i, cls_int]) # Original line

        targets.append(cls_label)
        target_acc.append(cls_label) # Changed to cls_label as per instruction

    submission = pd.DataFrame({
        "ID": sample_sub["ID"],
        "Target": targets,
        "Target_Accuracy": target_acc,
    })

    submission.to_csv(filename, index=False)
    print(f"Submission file written to: {filename}")

In [24]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Assuming generate_submission_file is defined as in cell a9c81468
# Assuming CONG_MAP_INV, X_sub, sample_sub are available and up-to-date

print("Generating submission files with corrected Target_Accuracy (class labels)...")

# --- Re-initialize and Re-fit Original XGBoost Models (from Af64mXqYndvK) ---
# Ensure the base models are created and fitted correctly, matching original intent
# Also handling potential fallback to RandomForestClassifier
try:
    # Try using XGBoost
    enter_model = XGBClassifier(
        objective="multi:softprob", num_class=4, eval_metric="mlogloss", max_depth=4,
        n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        random_state=42, n_jobs=-1,
    )
    exit_model = XGBClassifier(
        objective="multi:softprob", num_class=4, eval_metric="mlogloss", max_depth=4,
        n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        random_state=42, n_jobs=-1,
    )
except ImportError:
    # Fallback to RandomForest if XGBoost is not installed
    enter_model = RandomForestClassifier(
        n_estimators=400, max_depth=8, random_state=42, n_jobs=-1,
    )
    exit_model = RandomForestClassifier(
        n_estimators=400, max_depth=8, random_state=42, n_jobs=-1,
    )

print("Re-fitting original XGBoost/RandomForest models...")
enter_model.fit(X_enter, y_enter)
exit_model.fit(X_exit, y_exit)


# --- Re-initialize and Re-train LR, SVC, RandomForestClassifier Models (from 65a0f08e) ---
print("Re-fitting Logistic Regression, SVC, and RandomForest models...")
lr_enter = LogisticRegression(max_iter=1000, random_state=42)
lr_enter.fit(X_enter, y_enter)

lr_exit = LogisticRegression(max_iter=1000, random_state=42)
lr_exit.fit(X_exit, y_exit)

svc_enter = SVC(probability=True, random_state=42)
svc_enter.fit(X_enter, y_enter)

svc_exit = SVC(probability=True, random_state=42)
svc_exit.fit(X_exit, y_exit)

# RandomForestClassifier might have been refitted already in 482e5abf, but re-fitting here for consistency.
rf_enter = RandomForestClassifier(random_state=42)
rf_enter.fit(X_enter, y_enter)

rf_exit = RandomForestClassifier(random_state=42)
rf_exit.fit(X_exit, y_exit)


# --- Define expanded hyperparameter search space (from e9ff9a3c) ---
param_grid_xgb_expanded = {
    'max_depth': [3, 4, 5, 6, 7],
    'n_estimators': [200, 300, 400, 500, 600],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 5, 10]
}
print("Expanded hyperparameter search space for XGBoost defined within cell.")

# --- Perform RandomizedSearchCV (from e94b4e8f) ---
enter_model_base_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
exit_model_base_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

print("Performing RandomizedSearchCV for Enter model with expanded parameters...")
random_search_enter = RandomizedSearchCV(
    estimator=enter_model_base_random,
    param_distributions=param_grid_xgb_expanded,
    n_iter=100,
    scoring='f1_weighted',
    cv=3,
    n_jobs=-1,
    verbose=0,
    random_state=42
)
random_search_enter.fit(X_enter, y_enter)
best_params_enter_random = random_search_enter.best_params_

print("Performing RandomizedSearchCV for Exit model with expanded parameters...")
random_search_exit = RandomizedSearchCV(
    estimator=exit_model_base_random,
    param_distributions=param_grid_xgb_expanded,
    n_iter=100,
    scoring='f1_weighted',
    cv=3,
    n_jobs=-1,
    verbose=0,
    random_state=42
)
random_search_exit.fit(X_exit, y_exit)
best_params_exit_random = random_search_exit.best_params_

print("Best hyperparameters from Random Search obtained.")

# --- Train final optimized XGBoost models (from ce3c8182) ---
final_enter_model_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    **best_params_enter_random
)
print("Fitting final_enter_model_random...")
final_enter_model_random.fit(X_enter, y_enter)

final_exit_model_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    **best_params_exit_random
)
print("Fitting final_exit_model_random...")
final_exit_model_random.fit(X_exit, y_exit)
print("Final optimized XGBoost models trained.")


# Rebuild X_sub with advanced features before generating predictions
print("Rebuilding X_sub with advanced features...")
X_sub = build_features_for_submission(sample_sub, all_grouped, HIST_LEN, EMBARGO)
X_sub = X_sub.reindex(columns=FEATURE_COLS, fill_value=0)
print("X_sub rebuilt.")


# Regenerate submission for original XGBoost models
generate_submission_file(enter_model, exit_model, X_sub, sample_sub, CONG_MAP_INV, "submission.csv")

# Regenerate submission for Logistic Regression models
generate_submission_file(lr_enter, lr_exit, X_sub, sample_sub, CONG_MAP_INV, "submission_lr.csv")

# Regenerate submission for SVC models
generate_submission_file(svc_enter, svc_exit, X_sub, sample_sub, CONG_MAP_INV, "submission_svc.csv")

# Regenerate submission for RandomForestClassifier models
generate_submission_file(rf_enter, rf_exit, X_sub, sample_sub, CONG_MAP_INV, "submission_rf.csv")

# Regenerate submission for optimized XGBoost models
generate_submission_file(final_enter_model_random, final_exit_model_random, X_sub, sample_sub, CONG_MAP_INV, "submission_optimized.csv")

print("All submission files regenerated with class labels in Target_Accuracy.")

Generating submission files with corrected Target_Accuracy (class labels)...
Re-fitting original XGBoost/RandomForest models...
Re-fitting Logistic Regression, SVC, and RandomForest models...
Expanded hyperparameter search space for XGBoost defined within cell.
Performing RandomizedSearchCV for Enter model with expanded parameters...
Performing RandomizedSearchCV for Exit model with expanded parameters...
Best hyperparameters from Random Search obtained.
Fitting final_enter_model_random...
Fitting final_exit_model_random...
Final optimized XGBoost models trained.
Rebuilding X_sub with advanced features...
X_sub rebuilt.
Submission file written to: submission.csv
Submission file written to: submission_lr.csv
Submission file written to: submission_svc.csv
Submission file written to: submission_rf.csv
Submission file written to: submission_optimized.csv
All submission files regenerated with class labels in Target_Accuracy.


In [25]:
# Re-evaluate optimized 'enter' model to ensure variables are defined
y_enter_pred_optimized = final_enter_model.predict(X_enter)
f1_enter_optimized = f1_score(y_enter, y_enter_pred_optimized, average='weighted')
accuracy_enter_optimized = accuracy_score(y_enter, y_enter_pred_optimized)

# Re-evaluate optimized 'exit' model to ensure variables are defined
y_exit_pred_optimized = final_exit_model.predict(X_exit)
f1_exit_optimized = f1_score(y_exit, y_exit_pred_optimized, average='weighted')
accuracy_exit_optimized = accuracy_score(y_exit, y_exit_pred_optimized)


new_results = pd.DataFrame({
    'Model': [
        'Optimized XGBoost (Local)', 'Optimized XGBoost (Local)',
        'Optimized XGBoost (Platform)' # Adding the platform result
    ],
    'Target': [
        'Enter', 'Exit',
        'Overall' # Platform results are usually overall
    ],
    'F1 Score': [
        f1_enter_optimized, f1_exit_optimized,
        0.740776438 # User-provided platform F1
    ],
    'Accuracy': [
        accuracy_enter_optimized, accuracy_exit_optimized,
        0.781818181 # User-provided platform Accuracy
    ]
})

evaluation_results = pd.concat([evaluation_results, new_results], ignore_index=True)

display(evaluation_results)

,Model,Target,F1 Score,Accuracy
0,XGBoost,Enter,0.600159,0.684524
1,XGBoost,Exit,0.934535,0.956027
2,Logistic Regression,Enter,0.583533,0.646343
3,Logistic Regression,Exit,0.934535,0.956027
4,SVC,Enter,0.644747,0.705491
5,SVC,Exit,0.934535,0.956027
6,RandomForestClassifier,Enter,0.997746,0.997748
7,RandomForestClassifier,Exit,0.999839,0.999839
8,Optimized XGBoost (Local),Enter,0.894658,0.898220
9,Optimized XGBoost (Local),Exit,0.934535,0.956027


In [26]:
new_platform_results = pd.DataFrame({
    'Model': [
        'RandomForestClassifier (Platform)',
        'XGBoost (Platform)'
    ],
    'Target': [
        'Overall',
        'Overall'
    ],
    'F1 Score': [
        0.734229724, # User-provided platform F1 for RF
        0.721838821  # User-provided platform F1 for XGBoost
    ],
    'Accuracy': [
        0.779545454, # User-provided platform Accuracy for RF
        0.777272727  # User-provided platform Accuracy for XGBoost
    ]
})

evaluation_results = pd.concat([evaluation_results, new_platform_results], ignore_index=True)

display(evaluation_results)

,Model,Target,F1 Score,Accuracy
0,XGBoost,Enter,0.600159,0.684524
1,XGBoost,Exit,0.934535,0.956027
2,Logistic Regression,Enter,0.583533,0.646343
3,Logistic Regression,Exit,0.934535,0.956027
4,SVC,Enter,0.644747,0.705491
5,SVC,Exit,0.934535,0.956027
6,RandomForestClassifier,Enter,0.997746,0.997748
7,RandomForestClassifier,Exit,0.999839,0.999839
8,Optimized XGBoost (Local),Enter,0.894658,0.898220
9,Optimized XGBoost (Local),Exit,0.934535,0.956027


# Expand Hyperparameter Search Space for XGBoost

Define a new, more extensive hyperparameter search space (`param_grid_xgb_expanded`) for the XGBoost models.

To expand the hyperparameter search space for XGBoost models. This involves defining a new dictionary `param_grid_xgb_expanded` that contains a broader range of values for hyperparameters like `max_depth`, `n_estimators`, `learning_rate`, `subsample`, and `colsample_bytree`, and potentially introduces new parameters such as `gamma` or `min_child_weight` to allow for a more thorough tuning process.

In [27]:
param_grid_xgb_expanded = {
    'max_depth': [3, 4, 5, 6, 7],
    'n_estimators': [200, 300, 400, 500, 600],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2],
    'min_child_weight': [1, 5, 10]
}

print("Expanded hyperparameter search space for XGBoost defined.")

Expanded hyperparameter search space for XGBoost defined.


## Implement Advanced Feature Engineering

Modify the `make_feature_vector` function to create more sophisticated features. This could include adding: more statistical aggregates over the history window (e.g., median, variance, trend indicators), interaction terms between existing features (e.g., `signal_enc` and `tod_sin`), and potentially creating features related to the day of the week or month from `datetimestamp_start` if not already present. This step will also involve ensuring the `build_training_matrix` function correctly utilizes these new features.


In [28]:
def make_feature_vector(
    hist_df: pd.DataFrame,
    hist_len: int = HIST_LEN,
) -> dict:
    """
    Build features from a history window (already filtered to one view_label
    and sorted by time_segment_id). Real-time safe: uses ONLY:
      - signaling history
      - simple aggregates
      - (optionally) time-of-day of last observed segment
    """
    assert len(hist_df) > 0

    # Ensure sorted
    hist_df = hist_df.sort_values("time_segment_id")

    sig = hist_df["signal_enc"].values
    # If shorter than hist_len, left-pad with earliest value (real-time safe)
    if len(sig) < hist_len:
        pad_val = sig[0] if len(sig) > 0 else 0 # Ensure pad_val is defined even for empty hist_df (though assert should prevent this)
        sig = np.concatenate([np.full(hist_len - len(sig), pad_val), sig])
    else:
        sig = sig[-hist_len:]  # take most recent hist_len

    feats = {}

    # Raw lags: lag_1 = most recent pre-embargo, lag_hist_len = oldest in window
    # (this ordering is arbitrary but consistent)
    for i in range(hist_len):
        feats[f"sig_lag_{i+1}"] = int(sig[-(i+1)])

    # Aggregates
    feats["sig_mean_15"] = float(sig.mean())
    feats["sig_std_15"] = float(sig.std())
    feats["sig_min_15"] = float(sig.min())
    feats["sig_max_15"] = float(sig.max())
    feats["sig_median_15"] = float(np.median(sig)) # New: Median
    feats["sig_var_15"] = float(np.var(sig))     # New: Variance

    # Trend Indicators (New)
    # Difference between most recent and oldest in window
    feats["sig_trend_diff"] = float(sig[-1] - sig[0])
    # Simple linear regression slope (if window is large enough)
    if len(sig) > 1:
        x = np.arange(len(sig))
        slope = np.polyfit(x, sig, 1)[0]
        feats["sig_trend_slope"] = float(slope)
    else:
        feats["sig_trend_slope"] = 0.0

    # Time-of-day features from last history segment (if available)
    if "datetimestamp_start" in hist_df.columns and hist_df["datetimestamp_start"].notna().any():
        last_ts = hist_df["datetimestamp_start"].iloc[-1]
        if pd.notna(last_ts):
            hour = last_ts.hour + last_ts.minute / 60.0
            feats["tod_sin"] = np.sin(2 * np.pi * hour / 24.0)
            feats["tod_cos"] = np.cos(2 * np.pi * hour / 24.0)
            feats["day_of_week"] = last_ts.dayofweek # New: Day of week
            feats["month"] = last_ts.month         # New: Month
        else:
            feats["tod_sin"] = 0.0
            feats["tod_cos"] = 0.0
            feats["day_of_week"] = 0             # Default for missing
            feats["month"] = 0                # Default for missing
    else:
        feats["tod_sin"] = 0.0
        feats["tod_cos"] = 0.0
        feats["day_of_week"] = 0                 # Default if column missing
        feats["month"] = 0                    # Default if column missing

    # Interaction Terms (New)
    # Ensure 'sig_mean_15' and 'tod_sin' are present before creating interaction
    if 'sig_mean_15' in feats and 'tod_sin' in feats:
        feats["interaction_mean_tod_sin"] = feats["sig_mean_15"] * feats["tod_sin"]
    else:
        feats["interaction_mean_tod_sin"] = 0.0 # Default if components missing

    return feats

print("make_feature_vector function updated with advanced features.")

make_feature_vector function updated with advanced features.


In [29]:
print("Rebuilding training sets with advanced features...")

phases = ["train"] + (["test_input_15"] if USE_TEST_INPUT_AS_TRAIN else [])

X_enter, y_enter = build_training_matrix(
    all_df,
    label_col="enter_label_int",
    phases_for_training=phases,
    hist_len=HIST_LEN,
    embargo=EMBARGO,
)

X_exit, y_exit = build_training_matrix(
    all_df,
    label_col="exit_label_int",
    phases_for_training=phases,
    hist_len=HIST_LEN,
    embargo=EMBARGO,
)

# Update FEATURE_COLS to include the newly generated features
FEATURE_COLS = sorted(set(X_enter.columns) | set(X_exit.columns))
X_enter = X_enter.reindex(columns=FEATURE_COLS, fill_value=0)
X_exit = X_exit.reindex(columns=FEATURE_COLS, fill_value=0)

print("Training sets rebuilt with advanced features.")
print(f"New number of features: {len(FEATURE_COLS)}")

Rebuilding training sets with advanced features...
Training sets rebuilt with advanced features.
New number of features: 32


In [30]:
#from sklearn.model_selection import GridSearchCV

# Re-initialize the models to ensure they use the latest XGBClassifier (in case the try/except block was previously hit by RandomForestClassifier)
# Also, ensure the base models for GridSearchCV are not already fitted.
#enter_model_base = XGBClassifier(
#    objective="multi:softprob",
#    num_class=4,
#    eval_metric="mlogloss",
#    random_state=42,
#    n_jobs=-1,
#)
#exit_model_base = XGBClassifier(
#    objective="multi:softprob",
#    num_class=4,
#    eval_metric="mlogloss",
#    random_state=42,
#    n_jobs=-1,
#)

# Instantiate GridSearchCV for the 'enter' XGBoost model with expanded grid
#grid_search_enter_expanded = GridSearchCV(estimator=enter_model_base, param_grid=param_grid_xgb_expanded, scoring='f1_weighted', cv=3, n_jobs=-1, verbose=1)

# Fit GridSearchCV to the 'enter' training data with advanced features
#print("Performing GridSearchCV for Enter model with expanded parameters...")
#grid_search_enter_expanded.fit(X_enter, y_enter)

# Instantiate GridSearchCV for the 'exit' XGBoost model with expanded grid
#grid_search_exit_expanded = GridSearchCV(estimator=exit_model_base, param_grid=param_grid_xgb_expanded, scoring='f1_weighted', cv=3, n_jobs=-1, verbose=1)

# Fit GridSearchCV to the 'exit' training data with advanced features
#print("Performing GridSearchCV for Exit model with expanded parameters...")
#grid_search_exit_expanded.fit(X_exit, y_exit)

# Store the best hyperparameters from the expanded search
#best_params_enter_expanded = grid_search_enter_expanded.best_params_
#best_params_exit_expanded = grid_search_exit_expanded.best_params_

#print("\nBest hyperparameters for Enter model (Expanded Search):")
#print(best_params_enter_expanded)
#print("\nBest hyperparameters for Exit model (Expanded Search):")
#print(best_params_exit_expanded)

## Perform Hyperparameter Tuning with RandomizedSearchCV

Replace GridSearchCV with RandomizedSearchCV for both 'enter' and 'exit' XGBoost models. Set n_iter (e.g., 100) to limit the number of parameter combinations sampled, and retain cross-validation (cv=3) and parallel processing (n_jobs=-1). This will drastically reduce tuning time while still exploring a wide range of hyperparameters.


In [31]:
from sklearn.model_selection import RandomizedSearchCV

# Re-initialize the models to ensure they use the latest XGBClassifier
# Also, ensure the base models for RandomizedSearchCV are not already fitted.
enter_model_base_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)
exit_model_base_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

# Instantiate RandomizedSearchCV for the 'enter' XGBoost model with expanded grid
random_search_enter = RandomizedSearchCV(
    estimator=enter_model_base_random,
    param_distributions=param_grid_xgb_expanded,
    n_iter=100,  # Number of parameter settings that are sampled
    scoring='f1_weighted',
    cv=3,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit RandomizedSearchCV to the 'enter' training data with advanced features
print("Performing RandomizedSearchCV for Enter model with expanded parameters...")
random_search_enter.fit(X_enter, y_enter)

# Instantiate RandomizedSearchCV for the 'exit' XGBoost model with expanded grid
random_search_exit = RandomizedSearchCV(
    estimator=exit_model_base_random,
    param_distributions=param_grid_xgb_expanded,
    n_iter=100,  # Number of parameter settings that are sampled
    scoring='f1_weighted',
    cv=3,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit RandomizedSearchCV to the 'exit' training data with advanced features
print("Performing RandomizedSearchCV for Exit model with expanded parameters...")
random_search_exit.fit(X_exit, y_exit)

# Store the best hyperparameters from the expanded random search
best_params_enter_random = random_search_enter.best_params_
best_params_exit_random = random_search_exit.best_params_

print("\nBest hyperparameters for Enter model (Random Search):")
print(best_params_enter_random)
print("\nBest hyperparameters for Exit model (Random Search):")
print(best_params_exit_random)

Performing RandomizedSearchCV for Enter model with expanded parameters...
Fitting 3 folds for each of 100 candidates, totalling 300 fits
Performing RandomizedSearchCV for Exit model with expanded parameters...
Fitting 3 folds for each of 100 candidates, totalling 300 fits

Best hyperparameters for Enter model (Random Search):
{'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 7, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.8}

Best hyperparameters for Exit model (Random Search):
{'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}


## Train Optimized XGBoost Models with New Features (from Randomized Search)

Now to train the final 'enter' and 'exit' XGBoost models using the best hyperparameters found from RandomizedSearchCV, applied to the full training dataset with the newly engineered features.


In [32]:
from xgboost import XGBClassifier

# Instantiate and train the final 'enter' XGBoost model with best hyperparameters from RandomizedSearchCV
final_enter_model_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    **best_params_enter_random
)
print("Fitting final_enter_model_random...")
final_enter_model_random.fit(X_enter, y_enter)

# Instantiate and train the final 'exit' XGBoost model with best hyperparameters from RandomizedSearchCV
final_exit_model_random = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    **best_params_exit_random
)
print("Fitting final_exit_model_random...")
final_exit_model_random.fit(X_exit, y_exit)

print("Final XGBoost models (Randomized Search) trained with new features.")

Fitting final_enter_model_random...
Fitting final_exit_model_random...
Final XGBoost models (Randomized Search) trained with new features.


## Evaluate Optimized XGBoost Models using Cross-Validation

Evaluate the performance of the newly trained, optimized XGBoost models (from RandomizedSearchCV) using k-fold cross-validation (e.g., KFold(n_splits=5)) on the training data. Calculate and report the mean F1 score and accuracy across all folds for both 'enter' and 'exit' predictions to get a more reliable performance estimate.


In [33]:
from sklearn.model_selection import KFold, cross_val_score

# Instantiate KFold for cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# --- Evaluate 'enter' model ---
print("Evaluating Optimized Enter XGBoost Model with Cross-Validation...")

f1_scores_enter = cross_val_score(
    final_enter_model_random,
    X_enter,
    y_enter,
    cv=kf,
    scoring='f1_weighted',
    n_jobs=-1
)
accuracy_scores_enter = cross_val_score(
    final_enter_model_random,
    X_enter,
    y_enter,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)

print(f"Enter Model (Optimized XGBoost) - Mean F1 Score: {f1_scores_enter.mean():.4f} (+/- {f1_scores_enter.std():.4f})")
print(f"Enter Model (Optimized XGBoost) - Mean Accuracy: {accuracy_scores_enter.mean():.4f} (+/- {accuracy_scores_enter.std():.4f})")

# --- Evaluate 'exit' model ---
print("\nEvaluating Optimized Exit XGBoost Model with Cross-Validation...")

f1_scores_exit = cross_val_score(
    final_exit_model_random,
    X_exit,
    y_exit,
    cv=kf,
    scoring='f1_weighted',
    n_jobs=-1
)
accuracy_scores_exit = cross_val_score(
    final_exit_model_random,
    X_exit,
    y_exit,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)

print(f"Exit Model (Optimized XGBoost) - Mean F1 Score: {f1_scores_exit.mean():.4f} (+/- {f1_scores_exit.std():.4f})")
print(f"Exit Model (Optimized XGBoost) - Mean Accuracy: {accuracy_scores_exit.mean():.4f} (+/- {accuracy_scores_exit.std():.4f})")


Evaluating Optimized Enter XGBoost Model with Cross-Validation...
Enter Model (Optimized XGBoost) - Mean F1 Score: 0.7181 (+/- 0.0064)
Enter Model (Optimized XGBoost) - Mean Accuracy: 0.7299 (+/- 0.0077)

Evaluating Optimized Exit XGBoost Model with Cross-Validation...
Exit Model (Optimized XGBoost) - Mean F1 Score: 0.9345 (+/- 0.0025)
Exit Model (Optimized XGBoost) - Mean Accuracy: 0.9560 (+/- 0.0017)


# Develop an Ensemble Model

Create a simple ensemble by combining the predictions of the optimized XGBoost models (from RandomizedSearchCV) with RandomForestClassifier models. Use a blending approach by averaging predicted probabilities for both 'enter' and 'exit' predictions.


In [37]:
from sklearn.ensemble import RandomForestClassifier

# Re-train RandomForestClassifier models with the current, advanced feature set
rf_enter = RandomForestClassifier(random_state=42)
rf_enter.fit(X_enter, y_enter)

rf_exit = RandomForestClassifier(random_state=42)
rf_exit.fit(X_exit, y_exit)


# Rebuild X_sub with advanced features before generating predictions
print("Rebuilding X_sub with advanced features...")
X_sub = build_features_for_submission(sample_sub, all_grouped, HIST_LEN, EMBARGO)

# Ensure X_sub has the same columns and order as the training data
X_sub = X_sub.reindex(columns=FEATURE_COLS, fill_value=0)

print("Generating ensemble predictions...")

# 1. Get predicted probabilities from optimized XGBoost models
proba_xgb_enter = final_enter_model_random.predict_proba(X_sub)
proba_xgb_exit = final_exit_model_random.predict_proba(X_sub)

# 2. Get predicted probabilities from RandomForestClassifier models
proba_rf_enter = rf_enter.predict_proba(X_sub)
proba_rf_exit = rf_exit.predict_proba(X_sub)

targets_ensemble = []
target_acc_ensemble = []

# 3. and 4. Loop through sample_sub and blend predictions
for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        # Average probabilities for 'enter' predictions
        avg_proba = (proba_xgb_enter[i] + proba_rf_enter[i]) / 2
        # Determine the final predicted class
        cls_int = int(np.argmax(avg_proba))
        # Convert to label string
        cls_label = CONG_MAP_INV[cls_int]
    else:
        # Average probabilities for 'exit' predictions
        avg_proba = (proba_xgb_exit[i] + proba_rf_exit[i]) / 2
        # Determine the final predicted class
        cls_int = int(np.argmax(avg_proba))
        # Convert to label string
        cls_label = CONG_MAP_INV[cls_int]

    targets_ensemble.append(cls_label)
    target_acc_ensemble.append(cls_label) # Target_Accuracy contains the predicted class label

# 6. Create a pandas DataFrame for the submission
submission_ensemble_random = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_ensemble,
    "Target_Accuracy": target_acc_ensemble,
})

# 7. Save the DataFrame to a CSV file
submission_ensemble_random.to_csv("submission_ensemble_random.csv", index=False)
print("Submission file written to: submission_ensemble_random.csv")

Rebuilding X_sub with advanced features...
Generating ensemble predictions...
Submission file written to: submission_ensemble_random.csv


In [38]:
print("Rebuilding X_sub with advanced features...")
X_sub = build_features_for_submission(sample_sub, all_grouped, HIST_LEN, EMBARGO)

# Ensure X_sub has the same columns and order as the training data
X_sub = X_sub.reindex(columns=FEATURE_COLS, fill_value=0)

print("Generating ensemble predictions...")

# 1. Get predicted probabilities from optimized XGBoost models
proba_xgb_enter = final_enter_model_random.predict_proba(X_sub)
proba_xgb_exit = final_exit_model_random.predict_proba(X_sub)

# 2. Get predicted probabilities from RandomForestClassifier models
proba_rf_enter = rf_enter.predict_proba(X_sub)
proba_rf_exit = rf_exit.predict_proba(X_sub)

targets_ensemble = []
target_acc_ensemble = []

# 3. and 4. Loop through sample_sub and blend predictions
for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        # Average probabilities for 'enter' predictions
        avg_proba = (proba_xgb_enter[i] + proba_rf_enter[i]) / 2
        # Determine the final predicted class
        cls_int = int(np.argmax(avg_proba))
        # Convert to label string
        cls_label = CONG_MAP_INV[cls_int]
    else:
        # Average probabilities for 'exit' predictions
        avg_proba = (proba_xgb_exit[i] + proba_rf_exit[i]) / 2
        # Determine the final predicted class
        cls_int = int(np.argmax(avg_proba))
        # Convert to label string
        cls_label = CONG_MAP_INV[cls_int]

    targets_ensemble.append(cls_label)
    target_acc_ensemble.append(cls_label) # As per instruction, Target_Accuracy contains the predicted class label

# 6. Create a pandas DataFrame for the submission
submission_ensemble_random = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_ensemble,
    "Target_Accuracy": target_acc_ensemble,
})

# 7. Save the DataFrame to a CSV file
submission_ensemble_random.to_csv("submission_ensemble_random.csv", index=False)
print("Submission file written to: submission_ensemble_random.csv")

Rebuilding X_sub with advanced features...
Generating ensemble predictions...
Submission file written to: submission_ensemble_random.csv


In [39]:
from sklearn.ensemble import RandomForestClassifier

# Re-train RandomForestClassifier models
rf_enter = RandomForestClassifier(random_state=42)
rf_enter.fit(X_enter, y_enter)

rf_exit = RandomForestClassifier(random_state=42)
rf_exit.fit(X_exit, y_exit)


print("Rebuilding X_sub with advanced features...")
X_sub = build_features_for_submission(sample_sub, all_grouped, HIST_LEN, EMBARGO)

# Ensure X_sub has the same columns and order as the training data
X_sub = X_sub.reindex(columns=FEATURE_COLS, fill_value=0)

print("Generating ensemble predictions...")

# 1. Get predicted probabilities from optimized XGBoost models
proba_xgb_enter = final_enter_model_random.predict_proba(X_sub)
proba_xgb_exit = final_exit_model_random.predict_proba(X_sub)

# 2. Get predicted probabilities from RandomForestClassifier models
proba_rf_enter = rf_enter.predict_proba(X_sub)
proba_rf_exit = rf_exit.predict_proba(X_sub)

targets_ensemble = []
target_acc_ensemble = []

# 3. and 4. Loop through sample_sub and blend predictions
for i, row in sample_sub.iterrows():
    kind = row["target_kind"]

    if kind == "enter":
        # Average probabilities for 'enter' predictions
        avg_proba = (proba_xgb_enter[i] + proba_rf_enter[i]) / 2
        # Determine the final predicted class
        cls_int = int(np.argmax(avg_proba))
        # Convert to label string
        cls_label = CONG_MAP_INV[cls_int]
    else:
        # Average probabilities for 'exit' predictions
        avg_proba = (proba_xgb_exit[i] + proba_rf_exit[i]) / 2
        # Determine the final predicted class
        cls_int = int(np.argmax(avg_proba))
        # Convert to label string
        cls_label = CONG_MAP_INV[cls_int]

    targets_ensemble.append(cls_label)
    target_acc_ensemble.append(cls_label) # Target_Accuracy contains the predicted class label

# 6. Create a pandas DataFrame for the submission
submission_ensemble_random = pd.DataFrame({
    "ID": sample_sub["ID"],
    "Target": targets_ensemble,
    "Target_Accuracy": target_acc_ensemble,
})

# 7. Save the DataFrame to a CSV file
submission_ensemble_random.to_csv("submission_ensemble_random.csv", index=False)
print("Submission file written to: submission_ensemble_random.csv")

Rebuilding X_sub with advanced features...
Generating ensemble predictions...
Submission file written to: submission_ensemble_random.csv


In [40]:
import pandas as pd
import numpy as np

def generate_submission_file(enter_model, exit_model, X_sub, sample_sub, cong_map_inv, filename):
    """
    Generates a submission file with predicted class labels and their probabilities.

    Args:
        enter_model: Trained model for 'enter' congestion prediction.
        exit_model: Trained model for 'exit' congestion prediction.
        X_sub (pd.DataFrame): Test features DataFrame.
        sample_sub (pd.DataFrame): Sample submission DataFrame with parsed IDs.
        cong_map_inv (dict): Inverse mapping of congestion labels to integers.
        filename (str): Name of the output submission file.
    """
    # Predict class probabilities for ALL rows in X_sub
    enter_proba = enter_model.predict_proba(X_sub)
    exit_proba = exit_model.predict_proba(X_sub)

    targets = []
    target_acc = []

    for i, row in sample_sub.iterrows():
        kind = row["target_kind"]

        if kind == "enter":
            # Find the index of the class with the highest probability
            cls_int = int(np.argmax(enter_proba[i]))
            # Get the corresponding label string
            cls_label = cong_map_inv[cls_int]
            # Get the confidence (probability) of the predicted class
            # cls_conf = float(enter_proba[i, cls_int]) # Original line
        else:
            # Find the index of the class with the highest probability
            cls_int = int(np.argmax(exit_proba[i]))
            # Get the corresponding label string
            cls_label = cong_map_inv[cls_int]
            # Get the confidence (probability) of the predicted class
            # cls_conf = float(exit_proba[i, cls_int]) # Original line

        targets.append(cls_label)
        target_acc.append(cls_label) # Changed to cls_label as per instruction

    submission = pd.DataFrame({
        "ID": sample_sub["ID"],
        "Target": targets,
        "Target_Accuracy": target_acc,
    })

    submission.to_csv(filename, index=False)
    print(f"Submission file written to: {filename}")

print("Generating submission file for Optimized XGBoost (with new features from Randomized Search)...")
generate_submission_file(
    final_enter_model_random,
    final_exit_model_random,
    X_sub,
    sample_sub,
    CONG_MAP_INV,
    "submission_xgb_new_features_random.csv"
)
print("Submission file 'submission_xgb_new_features_random.csv' generated.")

Generating submission file for Optimized XGBoost (with new features from Randomized Search)...
Submission file written to: submission_xgb_new_features_random.csv
Submission file 'submission_xgb_new_features_random.csv' generated.


In [45]:
new_results_random_search = pd.DataFrame({
    'Model': ['Optimized XGBoost (Randomized Search)', 'Optimized XGBoost (Randomized Search)'],
    'Target': ['Enter', 'Exit'],
    'F1 Score': [f1_scores_enter.mean(), f1_scores_exit.mean()],
    'Accuracy': [accuracy_scores_enter.mean(), accuracy_scores_exit.mean()]
})


Now, I need to create the `ensemble_results` DataFrame by re-evaluating the performance of the ensemble model on the training data (`X_enter`, `y_enter`, `X_exit`, `y_exit`). This involves predicting probabilities from the individual models, averaging them, determining the final predicted classes, and then calculating the F1 score and accuracy for both 'enter' and 'exit' predictions.



In [46]:
from sklearn.metrics import f1_score, accuracy_score

# --- Evaluate Ensemble Model on Training Data ---

# 1. Predict probabilities for 'enter' ensemble on training data
proba_xgb_enter_train = final_enter_model_random.predict_proba(X_enter)
proba_rf_enter_train = rf_enter.predict_proba(X_enter)
avg_proba_enter_train = (proba_xgb_enter_train + proba_rf_enter_train) / 2
y_pred_ensemble_enter = np.argmax(avg_proba_enter_train, axis=1)

# Calculate F1 and accuracy for 'enter' ensemble
f1_ensemble_enter = f1_score(y_enter, y_pred_ensemble_enter, average='weighted')
accuracy_ensemble_enter = accuracy_score(y_enter, y_pred_ensemble_enter)

# 2. Predict probabilities for 'exit' ensemble on training data
proba_xgb_exit_train = final_exit_model_random.predict_proba(X_exit)
proba_rf_exit_train = rf_exit.predict_proba(X_exit)
avg_proba_exit_train = (proba_xgb_exit_train + proba_rf_exit_train) / 2
y_pred_ensemble_exit = np.argmax(avg_proba_exit_train, axis=1)

# Calculate F1 and accuracy for 'exit' ensemble
f1_ensemble_exit = f1_score(y_exit, y_pred_ensemble_exit, average='weighted')
accuracy_ensemble_exit = accuracy_score(y_exit, y_pred_ensemble_exit)

# Create DataFrame for ensemble results
ensemble_results = pd.DataFrame({
    'Model': ['Ensemble (XGBoost + RF)', 'Ensemble (XGBoost + RF)'],
    'Target': ['Enter', 'Exit'],
    'F1 Score': [f1_ensemble_enter, f1_ensemble_exit],
    'Accuracy': [accuracy_ensemble_enter, accuracy_ensemble_exit]
})

print("Ensemble model performance on training data calculated.")

Ensemble model performance on training data calculated.



Now that both `new_results_random_search` and `ensemble_results` DataFrames are created, I will concatenate them with the existing `evaluation_results` DataFrame and then display the updated table as instructed.



In [47]:
evaluation_results = pd.concat([evaluation_results, new_results_random_search, ensemble_results], ignore_index=True)

display(evaluation_results)

,Model,Target,F1 Score,Accuracy
0,XGBoost,Enter,0.600159,0.684524
1,XGBoost,Exit,0.934535,0.956027
2,Logistic Regression,Enter,0.583533,0.646343
3,Logistic Regression,Exit,0.934535,0.956027
4,SVC,Enter,0.644747,0.705491
5,SVC,Exit,0.934535,0.956027
6,RandomForestClassifier,Enter,0.997746,0.997748
7,RandomForestClassifier,Exit,0.999839,0.999839
8,Optimized XGBoost (Local),Enter,0.894658,0.898220
9,Optimized XGBoost (Local),Exit,0.934535,0.956027


In [48]:
import pandas as pd

# Re-initialize evaluation_results with initial XGBoost, LR, SVC, and RF results
evaluation_results = pd.DataFrame({
    'Model': [
        'XGBoost', 'XGBoost',
        'Logistic Regression', 'Logistic Regression',
        'SVC', 'SVC',
        'RandomForestClassifier', 'RandomForestClassifier'
    ],
    'Target': [
        'Enter', 'Exit',
        'Enter', 'Exit',
        'Enter', 'Exit',
        'Enter', 'Exit'
    ],
    'F1 Score': [
        f1_enter, f1_exit,
        f1_lr_enter, f1_lr_exit, # These were from a previous run, assuming they are defined
        f1_svc_enter, f1_svc_exit, # These were from a previous run, assuming they are defined
        f1_rf_enter, f1_rf_exit
    ],
    'Accuracy': [
        accuracy_enter, accuracy_exit,
        accuracy_lr_enter, accuracy_lr_exit, # Assuming defined
        accuracy_svc_enter, accuracy_svc_exit, # Assuming defined
        accuracy_rf_enter, accuracy_rf_exit
    ]
})

# Add previously added optimized XGBoost results (from old tuning)
new_optimized_xgb_results = pd.DataFrame({
    'Model': ['Optimized XGBoost', 'Optimized XGBoost'],
    'Target': ['Enter', 'Exit'],
    'F1 Score': [f1_enter_optimized, f1_exit_optimized], # Assuming these are defined from previous cells
    'Accuracy': [accuracy_enter_optimized, accuracy_exit_optimized]
})
evaluation_results = pd.concat([evaluation_results, new_optimized_xgb_results], ignore_index=True)

# Add previously added platform results
new_platform_results_1 = pd.DataFrame({
    'Model': [
        'Optimized XGBoost (Local)', 'Optimized XGBoost (Local)',
        'Optimized XGBoost (Platform)' # Adding the platform result
    ],
    'Target': [
        'Enter', 'Exit',
        'Overall' # Platform results are usually overall
    ],
    'F1 Score': [
        f1_enter_optimized, f1_exit_optimized,
        0.740776438 # User-provided platform F1
    ],
    'Accuracy': [
        accuracy_enter_optimized, accuracy_exit_optimized,
        0.781818181 # User-provided platform Accuracy
    ]
})

new_platform_results_2 = pd.DataFrame({
    'Model': [
        'RandomForestClassifier (Platform)',
        'XGBoost (Platform)'
    ],
    'Target': [
        'Overall',
        'Overall'
    ],
    'F1 Score': [
        0.734229724, # User-provided platform F1 for RF
        0.721838821  # User-provided platform F1 for XGBoost
    ],
    'Accuracy': [
        0.779545454, # User-provided platform Accuracy for RF
        0.777272727  # User-provided platform Accuracy for XGBoost
    ]
})

evaluation_results = pd.concat([evaluation_results, new_platform_results_1, new_platform_results_2], ignore_index=True)

# Now, add the new results from randomized search and ensemble
evaluation_results = pd.concat([evaluation_results, new_results_random_search, ensemble_results], ignore_index=True)

display(evaluation_results)


,Model,Target,F1 Score,Accuracy
0,XGBoost,Enter,0.600159,0.684524
1,XGBoost,Exit,0.934535,0.956027
2,Logistic Regression,Enter,0.583533,0.646343
3,Logistic Regression,Exit,0.934535,0.956027
4,SVC,Enter,0.644747,0.705491
5,SVC,Exit,0.934535,0.956027
6,RandomForestClassifier,Enter,0.997746,0.997748
7,RandomForestClassifier,Exit,0.999839,0.999839
8,Optimized XGBoost,Enter,0.894658,0.898220
9,Optimized XGBoost,Exit,0.934535,0.956027


In [51]:
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# --- Re-initialize and Re-train Original XGBoost Models ---
enter_model = XGBClassifier(
    objective="multi:softprob", num_class=4, eval_metric="mlogloss", max_depth=4,
    n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    random_state=42, n_jobs=-1,
)
exit_model = XGBClassifier(
    objective="multi:softprob", num_class=4, eval_metric="mlogloss", max_depth=4,
    n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    random_state=42, n_jobs=-1,
)

print("Re-fitting original XGBoost models...")
enter_model.fit(X_enter, y_enter)
exit_model.fit(X_exit, y_exit)

y_enter_pred = enter_model.predict(X_enter)
f1_enter = f1_score(y_enter, y_enter_pred, average='weighted')
accuracy_enter = accuracy_score(y_enter, y_enter_pred)

y_exit_pred = exit_model.predict(X_exit)
f1_exit = f1_score(y_exit, y_exit_pred, average='weighted')
accuracy_exit = accuracy_score(y_exit, y_exit_pred)

# --- Re-initialize and Re-train LR, SVC, RandomForestClassifier Models ---
print("Re-fitting Logistic Regression, SVC, and RandomForest models...")
lr_enter = LogisticRegression(max_iter=1000, random_state=42)
lr_enter.fit(X_enter, y_enter)
lr_exit = LogisticRegression(max_iter=1000, random_state=42)
lr_exit.fit(X_exit, y_exit)

svc_enter = SVC(probability=True, random_state=42)
svc_enter.fit(X_enter, y_enter)
svc_exit = SVC(probability=True, random_state=42)
svc_exit.fit(X_exit, y_exit)

rf_enter = RandomForestClassifier(random_state=42)
rf_enter.fit(X_enter, y_enter)
rf_exit = RandomForestClassifier(random_state=42)
rf_exit.fit(X_exit, y_exit)

lr_enter_pred = lr_enter.predict(X_enter)
f1_lr_enter = f1_score(y_enter, lr_enter_pred, average='weighted')
accuracy_lr_enter = accuracy_score(y_enter, lr_enter_pred)

lr_exit_pred = lr_exit.predict(X_exit)
f1_lr_exit = f1_score(y_exit, lr_exit_pred, average='weighted')
accuracy_lr_exit = accuracy_score(y_exit, lr_exit_pred)

svc_enter_pred = svc_enter.predict(X_enter)
f1_svc_enter = f1_score(y_enter, svc_enter_pred, average='weighted')
accuracy_svc_enter = accuracy_score(y_enter, svc_enter_pred)

svc_exit_pred = svc_exit.predict(X_exit)
f1_svc_exit = f1_score(y_exit, svc_exit_pred, average='weighted')
accuracy_svc_exit = accuracy_score(y_exit, svc_exit_pred)

rf_enter_pred = rf_enter.predict(X_enter)
f1_rf_enter = f1_score(y_enter, rf_enter_pred, average='weighted')
accuracy_rf_enter = accuracy_score(y_enter, rf_enter_pred)

rf_exit_pred = rf_exit.predict(X_exit)
f1_rf_exit = f1_score(y_exit, rf_exit_pred, average='weighted')
accuracy_rf_exit = accuracy_score(y_exit, rf_exit_pred)

# --- Re-initialize and Re-train the 'first' Optimized XGBoost Models ---
# Using initial XGBoost parameters since best_params_enter/exit from the first GridSearchCV were problematic.
final_enter_model = XGBClassifier(
    objective="multi:softprob", num_class=4, eval_metric="mlogloss", max_depth=4,
    n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    random_state=42, n_jobs=-1,
)
final_exit_model = XGBClassifier(
    objective="multi:softprob", num_class=4, eval_metric="mlogloss", max_depth=4,
    n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
    random_state=42, n_jobs=-1,
)

print("Re-fitting first optimized XGBoost models...")
final_enter_model.fit(X_enter, y_enter)
final_exit_model.fit(X_exit, y_exit)

y_enter_pred_optimized = final_enter_model.predict(X_enter)
f1_enter_optimized = f1_score(y_enter, y_enter_pred_optimized, average='weighted')
accuracy_enter_optimized = accuracy_score(y_enter, y_enter_pred_optimized)

y_exit_pred_optimized = final_exit_model.predict(X_exit)
f1_exit_optimized = f1_score(y_exit, y_exit_pred_optimized, average='weighted')
accuracy_exit_optimized = accuracy_score(y_exit, y_exit_pred_optimized)


# --- Reconstruct evaluation_results DataFrame ---
evaluation_results = pd.DataFrame({
    'Model': [
        'XGBoost', 'XGBoost',
        'Logistic Regression', 'Logistic Regression',
        'SVC', 'SVC',
        'RandomForestClassifier', 'RandomForestClassifier'
    ],
    'Target': [
        'Enter', 'Exit',
        'Enter', 'Exit',
        'Enter', 'Exit',
        'Enter', 'Exit'
    ],
    'F1 Score': [
        f1_enter, f1_exit,
        f1_lr_enter, f1_lr_exit,
        f1_svc_enter, f1_svc_exit,
        f1_rf_enter, f1_rf_exit
    ],
    'Accuracy': [
        accuracy_enter, accuracy_exit,
        accuracy_lr_enter, accuracy_lr_exit,
        accuracy_svc_enter, accuracy_svc_exit,
        accuracy_rf_enter, accuracy_rf_exit
    ]
})

new_optimized_xgb_results = pd.DataFrame({
    'Model': ['Optimized XGBoost', 'Optimized XGBoost'],
    'Target': ['Enter', 'Exit'],
    'F1 Score': [f1_enter_optimized, f1_exit_optimized],
    'Accuracy': [accuracy_enter_optimized, accuracy_exit_optimized]
})
evaluation_results = pd.concat([evaluation_results, new_optimized_xgb_results], ignore_index=True)

new_platform_results_1 = pd.DataFrame({
    'Model': [
        'Optimized XGBoost (Local)', 'Optimized XGBoost (Local)',
        'Optimized XGBoost (Platform)'
    ],
    'Target': [
        'Enter', 'Exit',
        'Overall'
    ],
    'F1 Score': [
        f1_enter_optimized, f1_exit_optimized,
        0.740776438 # User-provided platform F1
    ],
    'Accuracy': [
        accuracy_enter_optimized, accuracy_exit_optimized,
        0.781818181 # User-provided platform Accuracy
    ]
})

new_platform_results_2 = pd.DataFrame({
    'Model': [
        'RandomForestClassifier (Platform)',
        'XGBoost (Platform)'
    ],
    'Target': [
        'Overall',
        'Overall'
    ],
    'F1 Score': [
        0.734229724, # User-provided platform F1 for RF
        0.721838821  # User-provided platform F1 for XGBoost
    ],
    'Accuracy': [
        0.779545454, # User-provided platform Accuracy for RF
        0.777272727  # User-provided platform Accuracy for XGBoost
    ]
})

evaluation_results = pd.concat([evaluation_results, new_platform_results_1, new_platform_results_2], ignore_index=True)

# Now, add the new results from randomized search and ensemble
evaluation_results = pd.concat([evaluation_results, new_results_random_search, ensemble_results], ignore_index=True)

display(evaluation_results)


Re-fitting original XGBoost models...
Re-fitting Logistic Regression, SVC, and RandomForest models...
Re-fitting first optimized XGBoost models...


,Model,Target,F1 Score,Accuracy
0,XGBoost,Enter,0.727261,0.756274
1,XGBoost,Exit,0.952070,0.964232
2,Logistic Regression,Enter,0.591583,0.660393
3,Logistic Regression,Exit,0.934509,0.955974
4,SVC,Enter,0.561370,0.667793
5,SVC,Exit,0.934535,0.956027
6,RandomForestClassifier,Enter,0.998123,0.998123
7,RandomForestClassifier,Exit,0.999839,0.999839
8,Optimized XGBoost,Enter,0.727261,0.756274
9,Optimized XGBoost,Exit,0.952070,0.964232
